**Анализ рынка жилой недвижимости Санкт‑Петербурга и Ленинградской области для разработки бизнес‑стратегии агентства недвижимости** 

**Цель исследования:**
    
Помочь агентству недвижимости выйти на рынок Санкт‑Петербурга и Ленинградской области: определить перспективные сегменты недвижимости и оптимальные сезоны для маркетинговых кампаний.    

**Задачи исследования:**

- выявить наиболее привлекательные сегменты рынка недвижимости (по городам, типам жилья, характеристикам квартир) на основе времени активности объявлений;

- проанализировать сезонные тенденции рынка (определить периоды пиковой активности продавцов и покупателей);

- сформировать практические рекомендации для бизнес‑стратегии агентства;

- создать дашборд в DataLens по результатам анализа.

**Описание данных:**

**Таблица `advertisement`содержит информацию об объявлениях:**
    
`id` — идентификатор объявления (первичный ключ);

`first_day_exposition` — дата подачи объявления;

`days_exposition` — длительность нахождения объявления на сайте (в днях);

`last_price` — стоимость квартиры в объявлении, в руб.

**Таблица `flats` cодержит информацию о квартирах:**

`id` — идентификатор квартиры (первичный ключ, связан с первичным ключом id таблицы advertisement);

`city_id` — идентификатор города (внешний ключ, связан с city_id таблицы city);

`type_id` — идентификатор типа населённого пункта (внешний ключ, связан с type_id таблицы type);

`total_area` — общая площадь квартиры, в кв. метрах;

`rooms` — число комнат;

`ceiling_height` — высота потолка, в метрах;

`floors_total` — этажность дома, в котором находится квартира;

`living_area` — жилая площадь, в кв. метрах;

`floor` — этаж квартиры;

`is_apartment` — указатель, является ли квартира апартаментами (1 — является, 0 — не является);

`open_plan` — указатель, имеется ли в квартире открытая планировка (1 — открытая планировка квартиры, 0 — открытая планировка отсутствует);

`kitchen_area` — площадь кухни, в кв. метрах;

`balcony` — количество балконов в квартире;

`airports_nearest` — расстояние до ближайшего аэропорта, в метрах;

`parks_around3000` — число парков в радиусе трёх километров;

`ponds_around3000` — число водоёмов в радиусе трёх километров

**Таблица `city` содержит информацию о городах:**

`city_id` — идентификатор населённого пункта (первичный ключ);

`city` — название населённого пункта.

**Таблица `type` cодержит информацию о городах:**

`type_id` — идентификатор типа населённого пункта (первичный ключ);

`type` — название типа населённого пункта.

**План проекта:**
- Загрузить и предобработать данные
- Исследовательский анализ данных
- Сформулировать выводы и рекомендации
- Построить дашборд с основными метриками исследования

## Загрузка и предобработка данных

**Задача 1**. Временной интервал данных.

In [ ]:
# Объявление о продажах - мин и мах даты
SELECT 
    MIN(first_day_exposition) AS min,
    MAX(first_day_exposition) AS max
FROM real_estate.advertisement;

Данные представлены с конца ноября 2014 года по начало мая 2019 года/

**Задача 2**. Типы населенных пунктов.

In [ ]:
SELECT t.type AS "Тип населенного пункта",
COUNT(distinct c.city_id) AS "Кол-во населенных пунктов", 
COUNT(a.id) AS "Кол-во объявлений"
FROM real_estate.type as t
JOIN real_estate.flats as f on t.type_id = f.type_id
JOIN real_estate.advertisement as a on f.id = a.id
JOIN real_estate.city as c on f.city_id = c.city_id
GROUP BY t.type;

Данные содержат информацию о 10 типах населённых пунктов. Большая часть объявлений содержит информацию о недвижимости в городах (20 008 объявлений или около 85% всех объявлений). Садоводческое некоммерческое товарищество содержит всего одно объявление.

**Задача 3**. Время активности объявлений.

In [ ]:
SELECT 
    MIN(days_exposition),
    MAX(days_exposition),
    ROUND(AVG(days_exposition::numeric), 2),
    PERCENTILE_DISC(0.5) within group (order by days_exposition) AS pere
FROM real_estate.advertisement;

Минимальное - 1 день, максимальное - 1580 дней, среднее - 180.75 и медиана составляет 95 дней. Видим, что медиана отличается от среднего значения, что может говорить о наличии единичных высоких значений — выбросов. 
Результаты показывают, что половину объявлений сняли с публикации в течение 95 дней с момента публикации   


**Задача 4**. Доля снятых с публикаций объявлений.

In [ ]:
SELECT 
    ROUND(COUNT(days_exposition)*100.0/COUNT(*), 2) AS percentage
FROM real_estate.advertisement;

Около 86.55 % всех продаваемых объектов недвижимости могли быть проданы

**Задача 5**. Объявления Санкт-Петербурга

In [ ]:
SELECT
  COUNT(*) FILTER (WHERE c.city = 'Санкт-Петербург') AS spb_count, -- количество объявлений только Санкт-Петербург
  COUNT(*) FILTER (WHERE c.city != 'Санкт-Петербург') AS lenobl_count, -- количество объявлений для Ленинградской области
  COUNT(*) AS total_count, -- Общее кол-во объявлений
  ROUND(COUNT(*) FILTER (WHERE c.city = 'Санкт-Петербург') * 100.0 / COUNT(*), 2) AS spb_percentage,
  ROUND(COUNT(*) FILTER (WHERE c.city != 'Санкт-Петербург') * 100.0 / COUNT(*), 2) AS lenobl_percentage
FROM real_estate.advertisement AS a
JOIN real_estate.flats AS f ON a.id = f.id
JOIN real_estate.city AS c ON f.city_id = c.city_id;


Таким образом, 66.47 % всех продаваемых объектов недвижимости находятся в пределах Санкт-Петербурга

**Задача 6**. Стоимость квадратного метра

In [ ]:
WITH price_m2 AS (
    SELECT a.last_price/f.total_area as m2_price
    FROM real_estate.advertisement as a
    JOIN real_estate.flats as f on a.id = f.id
    WHERE f.total_area > 0 and a.last_price > 0
)
SELECT 
    ROUND(MIN(m2_price)::numeric, 2) AS min_price,
    ROUND(MAX(m2_price)::numeric, 2) AS max_price,
    ROUND(AVG(m2_price)::numeric, 2) AS avg_price,
    ROUND(PERCENTILE_DISC(0.5) within group (order by m2_price)::numeric, 2) AS median_price 
FROM price_m2;

Минимальное - 111.83 руб., максимальное - 1907500.00 руб., среднее - 99432.25 руб. и медиана составляет 95000 руб. Видим, что среднее значение близко к медианному, это может говорить или о том, что в данных нет выбросов или аномальных значений. Или возможно они есть в части как низких значений, так и высоких. Самое низкое значение за квадратный метр - 112 рублей, возможно,  представлены не в рублях, а в тысячах (потому что  слишком низкие значения). Высокие же значения вполне могут быть реальной стоимостью.

**Задача 7. Статистические показатели по основным параметрам, выраженных в числовых значениях**

Необходимо проверить корректность данных и подсчитать статистические показатели — минимальное и максимальное значения, среднее значение, медиану и 99 перцентиль по следующим количественным данным: общая площадь недвижимости, количество комнат и балконов, высота потолков, этаж.

In [ ]:
# Статистические показатели - общая площадь, этажи, высота потолков
SELECT 
    ROUND(MIN(total_area)::numeric, 2) as min_total_area,
    ROUND(MAX(total_area)::numeric, 2) as max_total_area,
    ROUND(AVG(total_area)::numeric, 2) as avg_total_area,
    ROUND(PERCENTILE_DISC(0.99) within group (order by total_area)::numeric, 2) as perc_total_area
FROM real_estate.flats;

SELECT 
    ROUND(MIN(floors_total)::numeric, 2) as min_floors_total,
    ROUND(MAX(floors_total)::numeric, 2) as max_floors_total,
    ROUND(AVG(floors_total)::numeric, 2) as avg_floors_total,
    ROUND(PERCENTILE_DISC(0.99) within group (order by floors_total)::numeric, 2) as perc_floors_total
FROM real_estate.flats;

SELECT 
    ROUND(MIN(floor)::numeric, 2) as min_floor,
    ROUND(MAX(floor)::numeric, 2) as max_floor,
    ROUND(AVG(floor)::numeric, 2) as avg_floor,
    ROUND(PERCENTILE_DISC(0.99) within group (order by floor)::numeric, 2) as perc_floor
FROM real_estate.flats;

SELECT 
    ROUND(MIN(rooms)::numeric, 2) as min_rooms,
    ROUND(MAX(rooms)::numeric, 2) as max_rooms,
    ROUND(AVG(rooms)::numeric, 2) as avg_rooms,
    ROUND(PERCENTILE_DISC(0.99) within group (order by rooms)::numeric, 2) as perc_rooms
FROM real_estate.flats;

SELECT 
    ROUND(MAX(ceiling_height)::numeric, 2) as max_ceiling_height
    ROUND(MIN(ceiling_height)::numeric, 2) as min_ceiling_height
FROM real_estate.flats


# Определим аномальные значения (выбросы) по значению перцентилей:
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats),
-- Найдём id объявлений, которые не содержат выбросы:
filtered_id AS(
    SELECT id
    FROM real_estate.flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL))
-- Выведем объявления без выбросов:
SELECT *
FROM real_estate.flats
WHERE id IN (SELECT * FROM filtered_id);

Мы видим, что аномально высокие значения есть практически в каждом столбце, кроме этажа квартиры (33 этаж). Так количество комнат, общая площадь, высота потолков и количество балконов содержат высокие значения, которые редко встречаются среди недвижимости в России. 
Сравнив максимальное значение с 99 перцентилем, отфильтруем данные. Убираем высокие значения, которые являются аномалиями. Также уберем и низкое значение высоты потолка (1 метр). 

**Задача 8. Время активности объявления.** Нам нужно определить какие типы квартир Санкт-Петербурга и городов Ленинградской области продаются быстро (объявления о них быстро снимаются), а какие - долго. 

In [ ]:
# 1 Определим аномальные значения (выбросы) по значению перцентилей:
WITH limits AS (
    SELECT
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats
),
# 2 Найдём id объявлений, которые не содержат выбросы:
filtered_id AS (
    SELECT id
    FROM real_estate.flats
    WHERE total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
                AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits))
            OR ceiling_height IS NULL)),
# 3. Определяем категории Санкт-Петербург и ЛенОбл, категории по дням активности, 
# находим основные параметры для анализа недвижимости
category_date AS (
    SELECT
        CASE WHEN c.city = 'Санкт-Петербург' THEN 1
            ELSE 2
        END AS region_,
        CASE
	        WHEN a.days_exposition IS NULL THEN 0
            WHEN a.days_exposition BETWEEN 1 AND 30 THEN 1
            WHEN a.days_exposition BETWEEN 31 AND 90 THEN 2
            WHEN a.days_exposition BETWEEN 91 AND 180 THEN 3
            WHEN a.days_exposition >= 181 THEN 4
        END AS days_exposition_category,
        a.last_price * 1.0 /f.total_area AS price_m2, # стоимость 1 кв метра
        f.total_area AS total_area, # общая площадь
        f.living_area AS living_area, # жилая площадь
        f.rooms AS rooms, # количество комнат
        f.ceiling_height AS ceiling_height, # высота потолка
        f.balcony AS balcony, # количество балконов
        f.kitchen_area AS kitchen_area, # площадь кухни 
        f.floors_total AS floors_total, # этажность дома
        f.is_apartment as is_apartment, # апартаменты
        f.open_plan as open_plan, # открытая планировка
        f.parks_around3000 AS parks_around3000, # число парков в радиусе трех км
        f.ponds_around3000 AS ponds_around3000, # число водоемов в радиусе трех км
        f.airports_nearest AS airports_nearest, # расстояние до ближайшего аэропорта
        EXTRACT (year from a.first_day_exposition) AS year, # год публикации
        t.type AS type_, # населенные пункты 
        f.id AS id # объявления 
    FROM real_estate.flats AS f
    JOIN real_estate.advertisement AS a ON f.id = a.id
    JOIN real_estate.type AS t ON f.type_id = t.type_id
    JOIN real_estate.city AS c ON f.city_id = c.city_id)
# 4. Все собираем в итог
SELECT
    CASE # присваиваем тип региону 
        WHEN region_ = 1 THEN 'Санкт-Петербург'
        ELSE 'ЛенОбл'
    END AS region,
    CASE # присваиваем тип периоду активности
	    WHEN days_exposition_category = 0 THEN 'другое'
        WHEN days_exposition_category = 1 THEN 'до месяца'
        WHEN days_exposition_category = 2 THEN 'до трех месяцев'
        WHEN days_exposition_category = 3 THEN 'до полугода'
        WHEN days_exposition_category = 4 THEN 'более полугода'
    END AS period_aktiv,
    COUNT(DISTINCT id) AS "Количество объявлений", # количество объявлений
    ROUND(COUNT(DISTINCT id) *100.0/ SUM(COUNT(*)) OVER (), 2) AS "Доля объявлений", # доля объявлений
    ROUND(AVG(price_m2)::numeric, 0) AS "Средняя стоимость 1 кв м", # средняя стоимость 1 кв метра
    ROUND(AVG(total_area)::numeric, 2) AS "Средняя общая площадь", # средняя общая площадь
    ROUND(AVG(living_area)::numeric, 2) AS "Средняя жилая площадь", # средняя жилая площадь
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rooms) AS "Медиана количества комнат", # медиана количества комнат
    ROUND(AVG(ceiling_height)::numeric, 2) AS "Средняя высота потолка", # средняя высота потолка 
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY balcony) AS "Медиана количества балконов", # медиана количества балконов 
    ROUND(AVG(kitchen_area)::numeric, 2) AS "Средняя площадь кухни", # средняя площадь кухни 
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY floors_total) AS "Медиана этажности дома", # медиана этажности дома
    ROUND(AVG(is_apartment::numeric) *100.0, 2) AS "Процент апартаментов",
    ROUND(AVG(open_plan::numeric) *100.0, 2) AS "Процент с открытой планировкой",
    ROUND(AVG(case when rooms = 0 then 1 else 0 end) * 100.0, 2) AS "Процент студий",
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY parks_around3000) AS "Медиана парков", # медиана число парков в радиусе трех км
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ponds_around3000) AS "Медиана водоемов", # медиана водоемов в радиусе трех км
    ROUND(AVG(airports_nearest)::numeric, 2) AS "Расстояние до аэропорта"
FROM category_date
WHERE
    id IN (SELECT id FROM filtered_id) # осущетствляем фильтрацию объявлений
    AND type_ = 'город' # учитываем только города
    AND year between 2015 and 2018 # учитываем полные годы
GROUP BY region, period_aktiv, days_exposition_category
ORDER by region DESC, days_exposition_category;

**Выводы:**

*1. Какие сегменты рынка недвижимости Санкт-Петербурга и городов Ленинградской области имеют наиболее короткие или длинные сроки активности объявлений?*

- Город Санкт-Петербург имеет самые короткие сроки активности - 2168 объявлений. Самые длинные сроки имеют объявления “более полугода” - 3581. Ленинградская область, здесь короткие сроки представлены “до месяца”- 397, а самые длинные сроки “до трех месяцев” (917) и “более полугода” (890) соответственно. 
- Вывод - пик активности в Ленобласти приходится на объявления со сроком от 31 до 180 дней. В Санкт-Петербурге количество объявлений увеличивается с ростом времени. Таким образом, в обоих регионах заметен тренд на длительные периоды продаж. 

*2. Какие характеристики недвижимости, включая площадь недвижимости, среднюю стоимость квадратного метра, количество комнат и балконов и другие параметры, влияют на время активности объявлений? Как эти зависимости варьируют между регионами?*

- По средней площади, заметно, что чем больше объект, тем дольше он продается как в Санкт-Петербурге, так и в Ленобласти (от 30,48 кв.м до 37.21 кв.м - Санкт-Петербург, от 27,42 кв.м до 31,9 кв.м - ЛенОбл).
- Стоимость, цена за кв.м. В СПб растет с увеличением срока продажи (110 568 - 115 457 руб/кв.м, при этом в Ленобласти ситуация противоположная - цена за кв. снижается (73 275 - 68 297 руб/кв.м). 
- Площадь кухни - заметно, что “в долгих объектах”  площадь кухни крупнее как в СПб, так и в ЛенОбл. Высота потолков в Санкт-Петербурге растет от коротких к длинным продажам (от 2,76 до 2, 83), в Ленобласти квартиры имеют одинаковую этажность. 
- Этажность в СПб - хорошо продаются дома 10 - 12 этажности, в Ленинградской области стабильно - пятиэтажки. 
- Количество комнат как для СПб, так и ЛенОбл – 2 комнаты. Стоит выделить в категории «другие» Санкт-Петербурга количество комнат составляет 3. Количество балконов для всех неизменно и составляет – 1 балкон
- Медианное число парков и водоемов не имеет существенной корреляции.
- Процент студий как в Ленобласти, так и в Санкт-Петербурге незначительный, скорее всего формат студий стал развиваться и набирать популярность позже данного исследования. При этом отметим, что в течение месяца студия в Санкт-Петербурге продается активнее всего.
- Общий вывод - в Санкт-Петербурге крупные и дорогие объекты (большая площадь и высокие потолки) продаются дольше. А в Ленобласти  “дорогие объекты” за 73 275 руб/кв.м (при этом средняя площадь таких квартир небольшая, всего 48,72 кв.м) продаются быстрее (“до месяцв”), возможно, в связи с ограниченным предложением хороших небольших премиум- объектов в области

*3. Есть ли различия между недвижимостью Санкт-Петербурга и Ленинградской области по полученным результатам?*
- Различия есть, так доля быстрых продаж (“до месяцв”) в Санкт-Петербурге выше, нежели в ЛенОбл. Стоимость за 1 кв.м тоже имеет значение, так в СПб цена растет от “быстрых” сделок до “долгих”. В Ленобласти цена наоборот снижается. Интересно отметить еще один факт, что средние сегменты активности объявлений в Ленобласти (от трех месяцев) доминирую и составляют  33% от всех объявлений ЛенОбл. При этом в СПб рыночная динамика продаж распределена более равномерно с уклоном в “долгие” продажи. 
- С точки зрения комфорта, то в СПб квартиры продаются с большей общей площадью, с большей площадью кухни и с более высокими потолками, нежели в ЛенОбл.


**Задача 9. Сезонность объявлений.** Цель — выявить периоды с повышенной активностью продавцов и покупателей, а также оценить характеристики недвижимости в разные сезоны.

Для исследования объявлений по периодам используем дату публикации объявления и дату снятия объявления с публикации (как
снятие объявление - значит продажа недвижимости). 

In [ ]:
# Устанавливаем локаль для русских названий месяцев
SET lc_time = 'ru_RU';

WITH limits AS ( # определим аномальные значения (выбросы) по значению перцентилей:
    SELECT
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats),

filtered_id AS ( # найдём id объявлений, которые не содержат выбросы:
    SELECT id
    FROM real_estate.flats
    WHERE total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
                AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits))
            OR ceiling_height IS NULL)),

stat_1 AS ( # основные показатели для анализа
    SELECT
        a.id,
        a.first_day_exposition, # дата подачи объявления 
        a.days_exposition, # длительность нахождения объявления
        a.last_price, # стоимость квартиры
        f.total_area, # общая площадь
        (a.first_day_exposition + a.days_exposition * INTERVAL '1 day')::DATE AS rem_date, # дата снятия объявления
        a.last_price*1.0 /f.total_area AS price_m2 # стоимость 1 кв.м
    FROM real_estate.advertisement AS a
    JOIN real_estate.flats AS f ON a.id = f.id
    JOIN real_estate.type AS t ON f.type_id = t.type_id
    JOIN real_estate.city AS c ON f.city_id = c.city_id
    WHERE
        f.id IN (SELECT id FROM filtered_id) # осуществляем фильтрацию объявлений
        AND t.type = 'город' # фильтация по населенному пункту город
        AND EXTRACT(YEAR FROM a.first_day_exposition) BETWEEN 2015 AND 2018), # учитываются только данные за период с 2015 по 2018

stat_pub AS ( # считаем метрики по публикации объявлений
    SELECT
        EXTRACT(MONTH FROM first_day_exposition) AS month_num, # выделяем номер месяца из даты публикации объявления
        TO_CHAR(first_day_exposition, 'TMmon') AS month_name, # преобразовываем месяцы с помощью функции to_char
        COUNT(*) AS publications_count, # кол-во публикаций
        ROUND(AVG(price_m2)::numeric, 2) AS avg_price_m2_pub, # средняя стоимость кв.м в опубликованных объявлениях
        ROUND(AVG(total_area)::numeric, 2) AS avg_total_area_pub, # средняя площадь в опубликованных объявлениях
        RANK() OVER (ORDER BY COUNT(*) DESC) AS publication_rank, # ранжируем опубликованные объявления
        SUM(COUNT(*)) OVER () AS total_publications  # общее количество публикаций
    FROM stat_1
    GROUP BY month_num, month_name),

stat_rem AS ( # считаем метрики по продажам объявлений
    SELECT
        EXTRACT(MONTH FROM rem_date) AS month_num, # выделяем номер месяца из даты снятых объявления
        TO_CHAR(rem_date, 'TMmon') AS month_name, # преобразовываем месяцы с помощью функции to_char
        COUNT(*) AS sales_count, # количество продаж в проданных объявлениях
        ROUND(AVG(price_m2)::numeric, 2) AS avg_price_m2_rem, # средняя стоимость за кв.м в проданных объявлениях
        ROUND(AVG(total_area)::numeric, 2) AS avg_total_area_rem, # средняя площадь в проданных объявлениях
        RANK() OVER (ORDER BY COUNT(*) DESC) AS sales_rank, # ранжируем снятые объявления
        SUM(COUNT(*)) OVER () AS total_sales  # общее количество продаж
    FROM stat_1
    WHERE rem_date IS NOT NULL  # берем только завершенные продажи
    GROUP BY month_num, month_name)

SELECT # собираем все вместе
    p.month_name AS "Месяц",
    p.publications_count AS "Кол-во публикаций",
    ROUND((p.publications_count * 100.0 / p.total_publications), 2) AS "Процент опубликованных",
    r.sales_count AS "Кол-во продаж",
    ROUND((r.sales_count * 100.0 / r.total_sales), 2) AS "Процент снятых",
    p.publication_rank AS "Ранг публикаций",
    r.sales_rank AS "Ранг продаж",
    p.avg_price_m2_pub AS "Ср. цена в опуб.объявлениях",
    p.avg_total_area_pub AS "Ср. площадь в опуб.объявлениях",
    r.avg_price_m2_rem AS "Ср.цена в продан.объявлениях",
    r.avg_total_area_rem AS "Ср.площадь в продан.объявлениях"
FROM stat_pub as p
JOIN stat_rem as r ON p.month_num = r.month_num
ORDER BY p.month_num;  # сортировка по номеру месяца

**Выводы:**

*1. В какие месяцы наблюдается наибольшая активность в публикации объявлений о продаже недвижимости? А в какие — по снятию? Это показывает динамику активности покупателей.*

- Наибольшая активность по публикациям о продаже недвижимости выявлена в ноябре (1569), октябре (1437) и февраль (1369). Активность по снятию зафиксирована в октябре (1360), ноябре (1301) и сентябре (1238). Таким образом, мы видим, что пик предложений осенне-зимний период, а пик спроса - осень (в сентябре, в октябре и в ноябре)


*2. Совпадают ли периоды активной публикации объявлений и периоды, когда происходит повышенная продажа недвижимости (по месяцам снятия объявлений)?*
- Видно, что пики продаж публикаций и продаж приходятся на осенние месяцы (в ноябре и октябре)  и зимние, однако, есть различия. Так в октябре начинается пик продаж, при этом он предшествует пику публикаций (ноябрь). 
- В январе есть продажи, но самый минимум публикаций. В феврале минимум продаж, но зато представлено много предложений о продаже недвижимости. 
- Летний период - падение как спроса, так и предложения о продаже недвижимости. 

*3. Как сезонные колебания влияют на среднюю стоимость квадратного метра и среднюю площадь квартир? Что можно сказать о зависимости этих параметров от месяца?*
- Средняя цена в опубликованных объявлениях представлена следующим образом: наибольшая цена - сентябрь (107 563), август (107 034), январь (106 106); наименьшая - март (102 430), май (102 465), апрель (102 632) Вывод - цены  весной -  ниже, чем осенью и зимой.
- Средняя площадь в опубликованных предложениях представлена следующим образом: максимальная  - сентябрь (61,04), апрель (60,6), июль (60,42); минимальная - июнь (58,37), декабрь (58,84), август (58,99).
- Средняя цена за кв.м. по проданным объявлениям - представлена в марте (106 832), декабре (105 504), январе (104 947); самая низкая цена в мае (99 724) и в августе (100 036). Вывод продажи к концу весны начало лето - низкие.
- Средняя площадь в проданных объявлениях, максимальная в феврале (61,12), в марте (60,37), июне (59,82); самая низкая площадь в ноябре (56,71), августе (56,83) и в январе (57,53). 
- Таким образом, зимой и осенью цены выше, конец весны и лето - цены становятся ниже. Площадь объектов в опубликованных объявлениях осенью и весной выше, а летом и зимой ниже. В проданных объектах площадь не показывает явной сезонности, но максимальна в феврале, марте. 



**Задача 10. Анализ рынка недвижимости Ленобласти.** Необходимо определить, в каких населённых пунктах Ленинградской области активнее всего продаётся недвижимость и какая именно. 

In [ ]:
WITH limits AS ( # определим аномальные значения (выбросы) по значению перцентилей:
    SELECT
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats),

filtered_id AS ( # найдём id объявлений, которые не содержат выбросы:
    SELECT id
    FROM real_estate.flats
    WHERE total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
                AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits))
            OR ceiling_height IS NULL)),

city_lo AS ( # рассчитваем основные метрики
    SELECT
        c.city, # название населенного пункта
        COUNT(a.id) AS count_total, # общее кол-во объявлений
        COUNT(a.days_exposition) AS count_sell,  # кол-во продаж
        AVG(a.last_price * 1.0/ f.total_area) AS avg_price_m2, # средняя стоимость 1 кв.м
        AVG(f.total_area) AS avg_total_area, # средняя общая площадь
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY a.days_exposition) AS median_days_exposition, # медиана по дням продаж
        AVG(a.days_exposition) AS avg_days_exposition, # среднее кол-во дней, за которое квартира продалась
        NTILE(4) OVER (ORDER BY AVG(a.days_exposition)) AS speed_rank # разделение на 4 категории по скорости продаж
    FROM real_estate.advertisement a
    JOIN real_estate.flats f ON a.id = f.id
    JOIN real_estate.city c ON f.city_id = c.city_id
    JOIN real_estate.type t ON f.type_id = t.type_id
    WHERE
        f.id IN (SELECT id FROM filtered_id) # осущетствляем фильтрацию объявлений
        AND c.city != 'Санкт-Петербург' # выбираем населенные пункты ленобласти, которые "не СПб"
    GROUP BY c.city), # группируем по населенным пунктам

filtered_city AS ( # присваиваем каждой категории скорости название
    SELECT *,
        CASE speed_rank
            WHEN 1 THEN 'Очень быстро'
            WHEN 2 THEN 'Быстро'
            WHEN 3 THEN 'Медленно'
            WHEN 4 THEN 'Очень медленно'
        END AS speed_category
    FROM city_lo
    WHERE count_total > 50) # отсекаем выбросы по малому количеству объявлений в населенном пункте ленобласти

SELECT # собираем итог
    city AS "Населённый пункт",
    count_total AS "Кол-во объявлений",
    count_sell AS "Количество продаж",
    ROUND((count_sell * 100.0 / count_total), 2) AS "Доля продаж, %",
    ROUND(avg_price_m2::numeric, 2) AS "Ср. цена кв.м, руб",
    ROUND(avg_total_area::numeric, 2) AS "Ср. площадь кв.м",
    ROUND(avg_days_exposition::numeric, 2) AS "Ср. кол-во дней продаж",
    ROUND(median_days_exposition::numeric, 2) AS "Медиана дней продаж",
    speed_rank AS "Ранг скорости",
    speed_category AS "Категория скорости"
FROM filtered_city
ORDER BY count_sell DESC;

**Выводы:**

*1. В каких населённые пунктах Ленинградской области наиболее активно публикуют объявления о продаже недвижимости?*
- Наиболее активно публикуют объявления в следующих населенных пунктах Ленобласти: Мурино (568), Кудрово (463), Шушары (404), Всеволожск (356), Паргалово (311).

*2. В каких населённых пунктах Ленинградской области — самая высокая доля снятых с публикации объявлений? Это может указывать на высокую долю продажи недвижимости.*
- Наибольшая доля продаж - Кудрово (93,74%), Мурино (93,66%), Тосно (93,1%), Шушары (92,57%) и Паргалово ( 92,6% ), Колпино (92,07%).


*3. Какова средняя стоимость одного квадратного метра и средняя площадь продаваемых квартир в различных населённых пунктах? Есть ли вариация значений по этим метрикам?*
- Средняя стоимость - цены разнообразны от 18 110, 43 (Сланцы) до 104 158,94 (Пушкин), разброс цен очень большой. При этом самые высокие цены в Пушкине (104 158,94) и Сестрорецке (103 848, 09), а низкие в Сланцах (18 110, 43 и Волхову (34 912,33). Площади варьируются от 42,16 кв.м. (Никольское) до  62,45 кв.м (Сестрорецк), разброс также значителен. При этом самые большие в Сестрорецке (62,45) и Пушкине (59,74), а самые маленькие в Никольском (42,16) и Мурино (43,86).


*4. Среди выделенных населённых пунктов какие пункты выделяются по продолжительности публикации объявлений? То есть где недвижимость продаётся быстрее, а где — медленнее?*
- Использовала результаты медианы по количеству дней продаж. Самые быстрые продажи - Сосновый Бор (45 дней), Кингисепп (45 дней), Янино-1 (55 дней), Бугры (65 дней), Кронштад (74 дня), Мурино (74,5 дня) Кудрово (73,5 дней). Самые медленные продажи  - Никольское (123 дня), Коммунар (126,5 дня), Пушкин (127 дней), Красное Село (135,5 дней), Ломоносов (134 дня).
- Общий вывод - наиболее перспективные рынки - это Кудрово и Мурино, здесь высокая активность объявлений, хорошая конверсия (93,74%) и в среднем умеренные продажи (73 - 74 дня).

## Выводы и рекомендации

*Важно обратить внимание на следующие различия:*

- Ценовой разрыв, средняя цена в СПб (примерно 110 - 115 тыс.руб.) в то время как Ленобласти (50 - 95 тыс.руб). 
- Доля рынка Санкт-Петербурга доминирует (объявлений о продажах в разы больше), но при этом ЛенОбласти есть перспективные ниши - это Кудрово, Мурино, Шушары (близкие районы к СПб).
- По сезонности стоит отметить, что пик продаж приходится на октябрь, ноябрь, сентябрь. Зимний период характеризуется минимальным предложением (декабрь -январь), но ростом цен (декабрь-январь). А поздняя весна - лучший период для покупок, так как цены минимальны. 

*Рекомендации:*
- для долгих объектов ввести динамическое ценообразование (скидки, акции и др.); 
- усилить маркетинг зимой, когда снижается уровень предложений; 
- для проблемных городов, где низкая конверсия продаж в Ленобласти (Волхов, Отрадное) разработать специальные программы; 
- усилить маркетинг в близлежащих населенных пунктов СПб - здесь и высокая ликвидность, и быстрые продажи (Кудрово, Мурино, Шушары, Сосновый Бор, Кингисепп); 
- в удаленных районах сфокусироваться на продажах с площадью 50+, которые хорошо продаются сейчас. 
- для объектов Санкт-Петербурга сформировать фокус на премиальные сегменты с улучшенными характеристиками. 


*ссылка на дашборд* - https://datalens.yandex/9sly2i488qtav
